In [2]:
import os
print(os.getcwd())

C:\dev\residential-energy-efficiency-stats\notebooks


In [3]:
import json

# Alternar idioma para los títulos/ejes/etiquetas de posibles gráficas posteriores (EN/ES)
LANG = 'ES'


def cargar_datos_edificio(filepath):
    """Carga la estructura JSON de la actividad."""
    with open(filepath, 'r', encoding='utf-8') as file:
        return json.load(file)


# Cargar el archivo de la actividad actual
# (Cambiar el nombre del JSON para probar Actividad 3 o Actividad 4)
ruta_archivo = '../data/Actividad3_input.json'
try:
    datos_edificio = cargar_datos_edificio(ruta_archivo)
    mensaje_carga = "Datos cargados correctamente desde" if LANG == 'ES' else "Data loaded successfully from"
    print(f"{mensaje_carga}: {ruta_archivo}")
except FileNotFoundError:
    error_carga = "Archivo no encontrado. Verifica la ruta." if LANG == 'ES' else "File not found. Check path."
    print(error_carga)

Datos cargados correctamente desde: ../data/Actividad3_input.json


In [4]:
# Tabla de coeficientes de simultaneidad según ITC-BT-10
# El índice del array equivale al número de viviendas (para N=1 hasta N=21)
coef_simultaneidad_tabla = [
    0, 1, 2, 3, 3.8, 4.6, 5.4, 6.2, 7.0, 7.8, 8.5,
    9.2, 9.9, 10.6, 11.3, 11.9, 12.5, 13.1, 13.7, 14.3, 14.8, 15.3
]

In [5]:
def calcular_carga_viviendas(datos):
    plantas = datos['viviendas']['plantas_destinadas']
    tipos = datos['viviendas']['tipos']

    total_viviendas = 0
    suma_potencias_individuales = 0

    viviendas_basicas = 0
    viviendas_elevadas = 0

    for tipo in tipos:
        # Cantidad total de este tipo de vivienda en todo el edificio
        cantidad_total = tipo['cantidad_por_planta'] * plantas
        total_viviendas += cantidad_total

        # Criterios REBT para Grado de Electrificación Elevada (mínimo 9200 W)
        sup_mayor_160 = tipo.get('superficie_m2', 0) > 160
        tiene_aa = tipo.get('aire_acondicionado', False)
        puntos_luz = tipo.get('puntos_alumbrado')
        muchos_puntos = puntos_luz is not None and puntos_luz > 30

        if sup_mayor_160 or tiene_aa or muchos_puntos:
            potencia_unitaria = 9200
            viviendas_elevadas += cantidad_total
        else:
            # Grado de Electrificación Básica (mínimo 5750 W)
            potencia_unitaria = 5750
            viviendas_basicas += cantidad_total

        suma_potencias_individuales += (potencia_unitaria * cantidad_total)

    # Cálculo del coeficiente de simultaneidad
    if total_viviendas <= 21:
        coeficiente = coef_simultaneidad_tabla[total_viviendas]
    else:
        coeficiente = 15.3 + (total_viviendas - 21) * 0.5

    # Cálculo de la potencia media armónica
    potencia_media = suma_potencias_individuales / total_viviendas if total_viviendas > 0 else 0

    # Previsión de carga total para el bloque de viviendas
    carga_total = coeficiente * potencia_media

    return {
        "total_viviendas": total_viviendas,
        "basicas": viviendas_basicas,
        "elevadas": viviendas_elevadas,
        "coeficiente": coeficiente,
        "potencia_media_W": potencia_media,
        "carga_total_W": carga_total
    }

# Ejecutar y mostrar resultados
if 'datos_edificio' in locals():
    res_viv = calcular_carga_viviendas(datos_edificio)

    if LANG == 'ES':
        print("--- RESULTADOS VIVIENDAS ---")
        print(f"Total de viviendas: {res_viv['total_viviendas']}")
        print(f"  - Grado Básico (5750 W): {res_viv['basicas']}")
        print(f"  - Grado Elevado (9200 W): {res_viv['elevadas']}")
        print(f"Coeficiente de simultaneidad aplicado: {res_viv['coeficiente']}")
        print(f"Potencia media por vivienda: {res_viv['potencia_media_W']:.2f} W")
        print(f"PREVISIÓN DE CARGA (VIVIENDAS): {res_viv['carga_total_W']:.2f} W")
    else:
        print("--- DWELLINGS RESULTS ---")
        print(f"Total dwellings: {res_viv['total_viviendas']}")
        print(f"  - Basic Level (5750 W): {res_viv['basicas']}")
        print(f"  - Elevated Level (9200 W): {res_viv['elevadas']}")
        print(f"Applied simultaneity factor: {res_viv['coeficiente']}")
        print(f"Average power per dwelling: {res_viv['potencia_media_W']:.2f} W")
        print(f"TOTAL LOAD FORECAST (DWELLINGS): {res_viv['carga_total_W']:.2f} W")

--- RESULTADOS VIVIENDAS ---
Total de viviendas: 40
  - Grado Básico (5750 W): 20
  - Grado Elevado (9200 W): 20
Coeficiente de simultaneidad aplicado: 24.8
Potencia media por vivienda: 7475.00 W
PREVISIÓN DE CARGA (VIVIENDAS): 185380.00 W


In [6]:
def calcular_carga_locales(datos):
    tipos_locales = datos['locales']['tipos']

    # Constantes reglamentarias (ITC-BT-10)
    POTENCIA_MINIMA_M2 = 100      # W/m²
    POTENCIA_MINIMA_LOCAL = 3450  # W

    carga_total_locales = 0
    detalles_locales = []

    for local in tipos_locales:
        tipo = local['tipo']
        cantidad = local['cantidad']
        superficie = local['superficie_m2']
        potencia_estimada = local.get('potencia_estimada_W') # Puede ser None

        # 1. Cálculo base por superficie
        potencia_por_superficie = superficie * POTENCIA_MINIMA_M2

        # 2. Garantizar el mínimo por local de 3450 W
        potencia_base = max(potencia_por_superficie, POTENCIA_MINIMA_LOCAL)

        # 3. Evaluar contra la potencia estimada si existe
        if potencia_estimada is not None:
            potencia_unitaria = max(potencia_base, potencia_estimada)
        else:
            potencia_unitaria = potencia_base

        # 4. Cálculo del subtotal por tipo
        potencia_subtotal = potencia_unitaria * cantidad
        carga_total_locales += potencia_subtotal

        detalles_locales.append({
            "tipo": tipo,
            "cantidad": cantidad,
            "superficie_m2": superficie,
            "potencia_unitaria_W": potencia_unitaria,
            "subtotal_W": potencia_subtotal
        })

    return {
        "carga_total_W": carga_total_locales,
        "detalles": detalles_locales
    }

# Ejecutar y mostrar resultados
if 'datos_edificio' in locals():
    res_locales = calcular_carga_locales(datos_edificio)

    if LANG == 'ES':
        print("--- RESULTADOS LOCALES COMERCIALES ---")
        for det in res_locales['detalles']:
            print(f"Tipo {det['tipo']} ({det['cantidad']} ud): {det['superficie_m2']} m² -> {det['potencia_unitaria_W']} W c/u (Subtotal: {det['subtotal_W']} W)")
        print("-" * 38)
        print(f"PREVISIÓN DE CARGA (LOCALES): {res_locales['carga_total_W']:.2f} W")
    else:
        print("--- COMMERCIAL PREMISES RESULTS ---")
        for det in res_locales['detalles']:
            print(f"Type {det['tipo']} ({det['cantidad']} units): {det['superficie_m2']} m² -> {det['potencia_unitaria_W']} W each (Subtotal: {det['subtotal_W']} W)")
        print("-" * 38)
        print(f"TOTAL LOAD FORECAST (PREMISES): {res_locales['carga_total_W']:.2f} W")

--- RESULTADOS LOCALES COMERCIALES ---
Tipo 1 (1 ud): 80 m² -> 8000 W c/u (Subtotal: 8000 W)
Tipo 2 (1 ud): 80 m² -> 8000 W c/u (Subtotal: 8000 W)
Tipo 3 (1 ud): 80 m² -> 8000 W c/u (Subtotal: 8000 W)
Tipo 4 (1 ud): 70 m² -> 7000 W c/u (Subtotal: 7000 W)
Tipo 5 (1 ud): 70 m² -> 9200 W c/u (Subtotal: 9200 W)
--------------------------------------
PREVISIÓN DE CARGA (LOCALES): 40200.00 W


Garajes y Vehiculos Electricos

In [7]:
def calcular_carga_garajes_ve(datos):
    # 1. Carga de Garajes (Iluminación y Ventilación)
    plantas_garaje = datos['garajes']['plantas']

    # Ratios reglamentarios (ITC-BT-10)
    W_M2_NATURAL = 10
    W_M2_FORZADA = 20

    carga_garajes = 0
    detalles_garaje = []

    for planta in plantas_garaje:
        sup = planta['superficie_m2']
        vent = planta['ventilacion'].lower()

        # Selección del ratio según el tipo de ventilación
        ratio = W_M2_FORZADA if vent == "forzada" else W_M2_NATURAL

        potencia_planta = sup * ratio
        carga_garajes += potencia_planta

        detalles_garaje.append({
            "planta": planta['planta'],
            "superficie_m2": sup,
            "ventilacion": vent,
            "ratio_W_m2": ratio,
            "potencia_W": potencia_planta
        })

    # 2. Carga para Vehículos Eléctricos
    datos_ve = datos['vehiculos_electricos']
    plazas_totales = datos_ve['plazas_totales']
    preve_SPL = datos_ve['preve_SPL']

    # El cálculo normativo toma el 10% de las plazas
    plazas_calculo = plazas_totales * 0.10
    potencia_base_ve = 3680 # W (16A a 230V)

    # Coeficiente según presencia de Sistema de Protección de la Línea (SPL)
    coef_SPL = 0.3 if preve_SPL else 1.0

    carga_ve = potencia_base_ve * plazas_calculo * coef_SPL

    return {
        "garajes": {
            "carga_total_W": carga_garajes,
            "detalles": detalles_garaje
        },
        "vehiculos_electricos": {
            "plazas_totales": plazas_totales,
            "plazas_calculo": plazas_calculo,
            "tiene_SPL": preve_SPL,
            "coef_aplicado": coef_SPL,
            "carga_total_W": carga_ve
        }
    }

# Ejecutar y mostrar resultados
if 'datos_edificio' in locals():
    res_garajes = calcular_carga_garajes_ve(datos_edificio)

    g_info = res_garajes['garajes']
    v_info = res_garajes['vehiculos_electricos']

    if LANG == 'ES':
        print("--- RESULTADOS GARAJES ---")
        for det in g_info['detalles']:
            print(f"Planta {det['planta']}: {det['superficie_m2']} m² (Ventilación {det['ventilacion']} a {det['ratio_W_m2']} W/m²) -> {det['potencia_W']} W")
        print(f"PREVISIÓN CARGA GARAJES: {g_info['carga_total_W']:.2f} W")

        print("\n--- RESULTADOS VEHÍCULOS ELÉCTRICOS (VE) ---")
        print(f"Plazas a prever (10% de {v_info['plazas_totales']}): {v_info['plazas_calculo']}")
        print(f"Sistema SPL: {'SÍ' if v_info['tiene_SPL'] else 'NO'} -> Coeficiente aplicado: {v_info['coef_aplicado']}")
        print(f"PREVISIÓN CARGA VE: {v_info['carga_total_W']:.2f} W")
    else:
        print("--- GARAGES RESULTS ---")
        for det in g_info['detalles']:
            print(f"Floor {det['planta']}: {det['superficie_m2']} m² ({det['ventilacion']} ventilation at {det['ratio_W_m2']} W/m²) -> {det['potencia_W']} W")
        print(f"TOTAL GARAGE LOAD: {g_info['carga_total_W']:.2f} W")

        print("\n--- ELECTRIC VEHICLES (EV) RESULTS ---")
        print(f"Forecasted spaces (10% of {v_info['plazas_totales']}): {v_info['plazas_calculo']}")
        print(f"SPL System: {'YES' if v_info['tiene_SPL'] else 'NO'} -> Applied coefficient: {v_info['coef_aplicado']}")
        print(f"TOTAL EV LOAD: {v_info['carga_total_W']:.2f} W")

--- RESULTADOS GARAJES ---
Planta -1: 600 m² (Ventilación forzada a 20 W/m²) -> 12000 W
Planta -2: 600 m² (Ventilación forzada a 20 W/m²) -> 12000 W
PREVISIÓN CARGA GARAJES: 24000.00 W

--- RESULTADOS VEHÍCULOS ELÉCTRICOS (VE) ---
Plazas a prever (10% de 50): 5.0
Sistema SPL: NO -> Coeficiente aplicado: 1.0
PREVISIÓN CARGA VE: 18400.00 W


Gemini said
Para completar la previsión de cargas, la siguiente celda calcula la demanda de los servicios generales.

En esta sección, se aplica el coeficiente de mayoración reglamentario del 1.3 (ITC-BT-47) para motores y equipos de elevación (ascensor y grupo de presión). Además, se incluye un diccionario de estimación de W/m² para el alumbrado según su tecnología, el cual puedes ajustar a los valores exactos definidos en tu clase.

Como paso final, he añadido dos celdas adicionales: una para sumar la Previsión de Carga Total del edificio y otra para generar una gráfica de distribución de cargas, aplicando la configuración bilingüe.

In [8]:
def calcular_carga_servicios_generales(datos):
    sg = datos['servicios_generales']

    # 1. Ascensor
    # Mapeo estándar de potencias nominales (ajustable según los criterios del problema)
    potencias_ascensor = {
        "ITA-1": 3000,
        "ITA-2": 4500,
        "ITA-3": 6000,
        "ITA-4": 7500
    }
    tipo_asc = sg['ascensor_tipo']
    potencia_nom_asc = potencias_ascensor.get(tipo_asc, 4500)

    # ITC-BT-47: Coeficiente de 1.3 para aparatos de elevación y motores
    COEF_MOTORES = 1.3
    carga_ascensor = potencia_nom_asc * COEF_MOTORES

    # 2. Grupo de Presión
    potencia_nom_grupo = sg['potencia_grupo_presion_W']
    carga_grupo = potencia_nom_grupo * COEF_MOTORES

    # 3. Alumbrado
    # Ratios estimados de W/m2 según eficiencia de la luminaria
    ratios_iluminacion = {
        "LED": 5,
        "incandescente": 15
    }

    alc = sg['alumbrado_zonas_comunes']
    ale = sg['alumbrado_escalera']

    # Extraer el tipo de luminaria respetando mayúsculas/minúsculas
    tipo_zc = "LED" if alc['tipo_luminaria'].upper() == "LED" else alc['tipo_luminaria'].lower()
    tipo_esc = "LED" if ale['tipo_luminaria'].upper() == "LED" else ale['tipo_luminaria'].lower()

    ratio_zc = ratios_iluminacion.get(tipo_zc, 10)
    ratio_esc = ratios_iluminacion.get(tipo_esc, 10)

    carga_alumbrado_zc = alc['superficie_m2'] * ratio_zc
    carga_alumbrado_esc = ale['superficie_m2'] * ratio_esc
    carga_alumbrado_total = carga_alumbrado_zc + carga_alumbrado_esc

    # Total Servicios Generales
    carga_total_sg = carga_ascensor + carga_grupo + carga_alumbrado_total

    return {
        "ascensor": {
            "tipo": tipo_asc,
            "nominal_W": potencia_nom_asc,
            "calculo_W": carga_ascensor
        },
        "grupo_presion": {
            "nominal_W": potencia_nom_grupo,
            "calculo_W": carga_grupo
        },
        "alumbrado": {
            "zc_W": carga_alumbrado_zc,
            "esc_W": carga_alumbrado_esc,
            "total_W": carga_alumbrado_total
        },
        "carga_total_W": carga_total_sg
    }



In [9]:
# Ejecutar y mostrar resultados
if 'datos_edificio' in locals():
    res_sg = calcular_carga_servicios_generales(datos_edificio)

    if LANG == 'ES':
        print("--- RESULTADOS SERVICIOS GENERALES ---")
        print(
            f"Ascensor ({res_sg['ascensor']['tipo']}): Nominal {res_sg['ascensor']['nominal_W']} W x 1.3 -> {res_sg['ascensor']['calculo_W']:.2f} W")
        print(
            f"Grupo de Presión: Nominal {res_sg['grupo_presion']['nominal_W']} W x 1.3 -> {res_sg['grupo_presion']['calculo_W']:.2f} W")
        print(f"Alumbrado Zonas Comunes y Escalera: {res_sg['alumbrado']['total_W']:.2f} W")
        print(f"PREVISIÓN CARGA SERVICIOS GENERALES: {res_sg['carga_total_W']:.2f} W")
    else:
        print("--- GENERAL SERVICES RESULTS ---")
        print(
            f"Elevator ({res_sg['ascensor']['tipo']}): Nominal {res_sg['ascensor']['nominal_W']} W x 1.3 -> {res_sg['ascensor']['calculo_W']:.2f} W")
        print(
            f"Water Pump: Nominal {res_sg['grupo_presion']['nominal_W']} W x 1.3 -> {res_sg['grupo_presion']['calculo_W']:.2f} W")
        print(f"Common Areas & Stairs Lighting: {res_sg['alumbrado']['total_W']:.2f} W")
        print(f"TOTAL GENERAL SERVICES LOAD: {res_sg['carga_total_W']:.2f} W")

--- RESULTADOS SERVICIOS GENERALES ---
Ascensor (ITA-3): Nominal 6000 W x 1.3 -> 7800.00 W
Grupo de Presión: Nominal 4000 W x 1.3 -> 5200.00 W
Alumbrado Zonas Comunes y Escalera: 1750.00 W
PREVISIÓN CARGA SERVICIOS GENERALES: 14750.00 W


Previsión de Carga Total del Edificio

In [16]:
# Suma final de todos los componentes previamente calculados
if all(v in globals() for v in ['res_viv', 'res_locales', 'res_garajes', 'res_sg']):
    carga_total_edificio = (
            res_viv['carga_total_W'] +
            res_locales['carga_total_W'] +
            res_garajes['garajes']['carga_total_W'] +
            res_garajes['vehiculos_electricos']['carga_total_W'] +
            res_sg['carga_total_W']
    )

    # Comprobar obligatoriedad de Centro de Transformación (CT)
    requiere_CT = carga_total_edificio > 100000

    if LANG == 'ES':
        print("\n" + "=" * 45)
        print("PREVISIÓN DE CARGA TOTAL DEL EDIFICIO")
        print("=" * 45)
        print(f"Total: {carga_total_edificio:.2f} W")
        print(f"¿Requiere espacio para Centro de Transformación (>100 kW)?: {'SÍ' if requiere_CT else 'NO'}")
    else:
        print("\n" + "=" * 45)
        print("TOTAL BUILDING LOAD FORECAST")
        print("=" * 45)
        print(f"Total: {carga_total_edificio:.2f} W")
        print(f"Requires space for Transformer Substation (>100 kW)?: {'YES' if requiere_CT else 'NO'}")


PREVISIÓN DE CARGA TOTAL DEL EDIFICIO
Total: 282730.00 W
¿Requiere espacio para Centro de Transformación (>100 kW)?: SÍ


Visualización de la Distribución de Cargas

In [ ]:
import matplotlib.pyplot as plt

if 'carga_total_edificio' in locals():
    # Configuración de textos según idioma
    if LANG == 'ES':
        title = 'Distribución de la Demanda de Potencia del Edificio'
        labels = ['Viviendas', 'Locales', 'Garajes', 'Vehículos Eléctricos', 'Servicios Generales']
        ylabel = 'Potencia (W)'
        annotation_text = f"Carga Total:\n{carga_total_edificio / 1000:.1f} kW"
    else:
        title = 'Building Power Demand Distribution'
        labels = ['Dwellings', 'Commercial', 'Garages', 'Electric Vehicles', 'General Services']
        ylabel = 'Power (W)'
        annotation_text = f"Total Load:\n{carga_total_edificio / 1000:.1f} kW"

    # Datos
    cargas = [
        res_viv['carga_total_W'],
        res_locales['carga_total_W'],
        res_garajes['garajes']['carga_total_W'],
        res_garajes['vehiculos_electricos']['carga_total_W'],
        res_sg['carga_total_W']
    ]

    # Generación de la gráfica
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(labels, cargas, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])

    # Personalización
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Anotaciones
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, yval + (max(cargas) * 0.01),
                f"{yval / 1000:.1f} kW", ha='center', va='bottom', fontsize=10)

    ax.text(0.95, 0.95, annotation_text, transform=ax.transAxes,
            fontsize=12, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()

Esta celda automatiza la determinación de la Caja General de Protección (CGP). Primero calcula la intensidad de corriente trifásica ($I = \frac{P}{\sqrt{3} \cdot V}$) a partir de la carga_total_edificio calculada en la celda 6, y luego escoge el calibre estandarizado inmediatamente superior dentro de las tablas normativas (como el de 630 A). También asigna automáticamente el esquema tipo 9 estipulado por las normas particulares para acometidas de esta envergadura.

In [17]:
import math

if 'carga_total_edificio' in globals():
    # Cálculo de la intensidad de corriente asumiendo red trifásica (400V línea-línea) y cos(phi) = 1
    voltaje_trifasico = 400
    intensidad_calculada = carga_total_edificio / (math.sqrt(3) * voltaje_trifasico)

    # Calibres estándar de fusibles para CGP según Normas Particulares (A)
    calibres_estandar = [100, 160, 250, 400, 630]

    # Seleccionar el calibre superior más próximo
    calibre_cgp = next((c for c in calibres_estandar if c >= intensidad_calculada), None)

    # Esquema interior requerido (tipo 9 para estas intensidades/acometidas subterráneas)
    esquema_cgp = 9

    if calibre_cgp:
        designacion_cgp = f"CGP-{esquema_cgp}-{calibre_cgp}A"
    else:
        designacion_cgp = "Requiere configuración especial / Múltiples CGP"

    # Resultados respetando el switch de idioma
    if LANG == 'ES':
        print("\n" + "="*45)
        print("DETERMINACIÓN DE LA CGP")
        print("="*45)
        print(f"Potencia Máxima Total: {carga_total_edificio:.2f} W")
        print(f"Intensidad de cálculo (a 400V): {intensidad_calculada:.2f} A")
        print(f"Calibre estandarizado seleccionado: {calibre_cgp} A")
        print(f"Esquema interior designado: {esquema_cgp}")
        print(f"RESULTADO: {designacion_cgp}")
        if intensidad_calculada > 630:
            print("AVISO: La intensidad supera el calibre máximo (630A). Consultar con compañía suministradora.")
    else:
        print("\n" + "="*45)
        print("GENERAL PROTECTION BOX (CGP) DETERMINATION")
        print("="*45)
        print(f"Total Maximum Power: {carga_total_edificio:.2f} W")
        print(f"Calculated Current (at 400V): {intensidad_calculada:.2f} A")
        print(f"Selected Standard Rating: {calibre_cgp} A")
        print(f"Designated Internal Scheme: {esquema_cgp}")
        print(f"RESULT: {designacion_cgp}")
        if intensidad_calculada > 630:
            print("WARNING: Current exceeds maximum rating (630A). Consult utility company.")
else:
    error_msg = "Error: Calcula la carga total (Celda 6) antes de determinar la CGP." if LANG == 'ES' else "Error: Calculate total load (Cell 6) before determining CGP."
    print(error_msg)


DETERMINACIÓN DE LA CGP
Potencia Máxima Total: 282730.00 W
Intensidad de cálculo (a 400V): 408.09 A
Calibre estandarizado seleccionado: 630 A
Esquema interior designado: 9
RESULTADO: CGP-9-630A
